# Phase 2 · Legal Named Entity Recognition — Architecture-2 encoder, full fine-tuning + word features + BiLSTM + CRF

| | |
|---|---|
| **Codabench** | Strict Macro-F1 **0.4246** · Strict Macro-P 0.4839 · Strict Macro-R 0.3919 (rank 4 of 4 in Phase 2) |
| **Own dev / validation** | fold-2 strict macro-F1 0.3317 (P 0.3795, R 0.3055), best epoch 8 of stage A |
| **Backbone** | Architecture-2 encoder-decoder (150M) |

**The task, in simple words:** find the important names in a judgment — appellant, respondent, judge, court, statute, precedent, date, … — as exact character positions with a label.

**How this notebook solves it:**
1. Split every document into words, remembering their exact character positions.
2. Read the words with the Phase-1 model, in overlapping windows (documents are longer than the model's context).
3. Give every word a tag: **B** = begins an entity, **I** = inside one, **O** = outside; the head makes sure the tag sequence is valid.
4. Turn the tags back into `[start, end, label]` spans.
5. Train on fold 1 and check on fold 2; then retrain on both folds and predict the test fold.

**What is special in this run:** **Full fine-tuning** — every weight of the Phase-1 model is trained, with a small learning rate.

*Exact setting:* full fine-tuning of the Architecture-2 encoder (all 6 layers + embeddings).

**To run it:** Attach the tokenizer notebook's output and the Phase-1 checkpoint as Kaggle inputs, add the Hugging Face token as the Kaggle secret **`HF`**, pick a GPU (T4), then *Run All*. Paths are in the configuration cell; if a checkpoint is not found there, the notebook searches `/kaggle/input`. Test labels are never used.

Every step below starts with a short explanation: first in simple words, then how it works exactly.

### CELL 1 · Imports + reproducibility + device

**In simple words:** Load the Python libraries, fix the random seed (so a re-run gives the same result) and pick the GPU.

**How it works:** Seeds for `random`, NumPy and PyTorch; `DEVICE` is the GPU when one is attached.

In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


### CELL 2 · Configuration

**In simple words:** Every setting of this run in one place: input paths, sizes, learning rates, epochs. To change the experiment, change only this cell.

**How it works:** The comment block at the top summarises the whole method. Mixed precision: float16 + GradScaler on T4 / P100, bfloat16 on A100. `DEBUG_MODE = True` (where present) runs a few-minute check on a small subset.

In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# E1: Architecture-2 encoder (full fine-tuning) + word features + BiLSTM + CRF
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 Architecture-2 checkpoint. If wrong, CELL 11 searches /kaggle/input for it.
CHECKPOINT_PATH = Path(
    "/kaggle/input/models/ritikguptatab/proj-cap-p-1-archi-2/"
    "pytorch/default/1/best_encoder_decoder.pt"
)

OUTPUT_DIR = Path("/kaggle/working/e1_ner_arch2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "best_e1_fold1.pt"        # stage A (fold1 -> fold2)
FINAL_MODEL_PATH = OUTPUT_DIR / "final_e1_all_folds.pt"  # stage B (fold1 + fold2)


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

TRAIN_FOLDS = ["fold1"]          # stage A: train
VAL_FOLDS = ["fold2"]            # stage A: validate (score for the report)

# Stage B: retrain from the Phase-1 weights on fold1 + fold2 for the best
# number of epochs found in stage A, then predict the test set.
# (Twice the training data -> better test score.)
FINAL_TRAIN_ON_ALL_FOLDS = True


# ------------------------------------------------------------
# Windows (the encoder was pretrained on 1024-token inputs)
# ------------------------------------------------------------

WINDOW = 512
STRIDE = 384


# ------------------------------------------------------------
# Head
# ------------------------------------------------------------

SHAPE_DIM = 32           # word-shape embedding ("Xxxx", "dd.dd.dddd", ...)
LSTM_HIDDEN = 256        # per direction
HEAD_DROPOUT = 0.3


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

EPOCHS = 15
BATCH_SIZE = 8                   # windows per step. If CUDA out-of-memory: 4
EVAL_BATCH_SIZE = 16

LR_ENCODER = 1e-4                # pretrained encoder
LR_HEAD = 2e-3                   # word features + BiLSTM + CRF (new)
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
PATIENCE = 4                     # stage A early stopping (epochs without val improvement)
TRAIN_EVAL_DOCS = 8              # training docs scored each epoch (under/overfitting check)


# ------------------------------------------------------------
# Debug mode: tiny run to check the whole pipeline
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 12

if DEBUG_MODE:
    EPOCHS = 2


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


print("Stage A:", TRAIN_FOLDS, "->", VAL_FOLDS, "| Stage B (all folds):", FINAL_TRAIN_ON_ALL_FOLDS)
print("Window / stride:", WINDOW, "/", STRIDE)
print("AMP dtype:", AMP_DTYPE if USE_AMP else "disabled")

Stage A: ['fold1'] -> ['fold2'] | Stage B (all folds): True
Window / stride: 512 / 384
AMP dtype: torch.float16


### CELL 3 · Load tokenizer + tokenize with character offsets

**In simple words:** NER answers must be exact character positions, so while tokenizing we also remember where every token starts and ends in the original text.

**How it works:** Same pre-tokenizer and BPE merges as `tokenizer.py`, plus a `(start, end)` character offset for every token.

In [3]:
# ============================================================
# CELL 3 — LOAD TOKENIZER + TOKENIZE WITH CHARACTER OFFSETS
#
# NER must return CHARACTER positions, so for every token we
# also need the exact start / end character in the ORIGINAL text.
# We use the same pre-tokenizer and BPE as tokenizer.py, but
# keep track of positions.
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


bpe_cache = {}      # piece -> list of BPE tokens (big speed-up: words repeat a lot)


def bpe_tokens(piece):

    if piece not in bpe_cache:
        bpe_cache[piece] = tokenizer._apply_bpe(list(piece))

    return bpe_cache[piece]


def encode_with_offsets(text):
    """
    Returns four lists (one entry per token):
        ids    - token ids (identical to tokenizer.encode(text))
        starts - start character in the ORIGINAL text
        ends   - end character (exclusive) in the ORIGINAL text
        words  - index of the word (pre-tokenizer piece) the token belongs to
    """

    normalized = tok_module.normalize_text(text)

    if normalized == text:

        work_text = text          # normal case: positions are unchanged
        origin = None

    else:

        # NFKC can change the text length (for example "ﬁ" -> "fi").
        # Normalize character by character and remember where each
        # character came from.
        chars = []
        origin = []

        for i, ch in enumerate(text):
            n = tok_module.normalize_text(ch)
            chars.append(n)
            origin.extend([i] * len(n))

        work_text = "".join(chars)

    ids, starts, ends, words = [], [], [], []

    for word_index, match in enumerate(tok_module.PRETOKEN_PATTERN.finditer(work_text)):

        position = match.start()

        for token in bpe_tokens(match.group()):

            token_id = tokenizer.vocab.get(token)

            # (token, start in work_text) pieces; unknown tokens fall back to characters
            if token_id is not None:
                parts = [(token_id, position, position + len(token))]
            else:
                parts = [
                    (tokenizer.vocab.get(ch, UNK_ID), position + k, position + k + 1)
                    for k, ch in enumerate(token)
                ]

            for part_id, s, e in parts:

                ids.append(part_id)
                words.append(word_index)

                if origin is None:
                    starts.append(s)
                    ends.append(e)
                else:
                    starts.append(origin[s])
                    ends.append(origin[e - 1] + 1)

            position += len(token)

    return ids, starts, ends, words


# ------------------------------------------------------------
# Sanity test
# ------------------------------------------------------------

demo = "The Supreme Court of India held that Section 12 applies."

demo_ids, demo_s, demo_e, demo_w = encode_with_offsets(demo)

assert demo_ids == tokenizer.encode(demo)

print("\nToken           start end  word  text")

for i in range(min(10, len(demo_ids))):
    print(f"  {tokenizer.id_to_token[demo_ids[i]]!r:14} {demo_s[i]:4d} {demo_e[i]:4d} {demo_w[i]:4d}  {demo[demo_s[i]:demo_e[i]]!r}")

Vocabulary size: 32004 | PAD / UNK: 0 1

Token           start end  word  text
  'The'             0    3    0  'The'
  ' '               3    4    1  ' '
  'Supreme'         4   11    2  'Supreme'
  ' '              11   12    3  ' '
  'Court'          12   17    4  'Court'
  ' '              17   18    5  ' '
  'of'             18   20    6  'of'
  ' '              20   21    7  ' '
  'India'          21   26    8  'India'
  ' '              26   27    9  ' '


### CELL 4 · Download Phase-2 data (needs the Kaggle secret "HF")

**In simple words:** Download the official data of this phase from Hugging Face.

**How it works:** The dataset is gated: the token comes from the Kaggle secret `HF` (never written in the notebook). Files are cached, so a re-run does not download again.

In [4]:
# ============================================================
# CELL 4 — DOWNLOAD PHASE-2 DATA  (needs the Kaggle secret "HF")
# ============================================================

from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")

PHASE2_FILES = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "labels":     "Phase_2_LNER/Train/labels.txt",
    "test_text":  "Phase_2_LNER/Test/fold3-text.json",
}

DATA_FILES = {}

for name, filename in PHASE2_FILES.items():

    DATA_FILES[name] = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    print(f"{name:11s} -> {DATA_FILES[name]}")

fold1_text  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json


fold1_gold  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json


fold2_text  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json


fold2_gold  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json


labels      -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt


test_text   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Test/fold3-text.json


### CELL 5 · Load texts, labels and gold entities

**In simple words:** Read the documents of fold 1, fold 2 and the test fold, the 12 entity labels, and the gold entities.

**How it works:** Every gold entity is `(start, end, label)` with end-exclusive character offsets.

In [5]:
# ============================================================
# CELL 5 — LOAD TEXTS, LABELS AND GOLD ENTITIES
# ============================================================

def load_texts(path):
    """{doc_id: text}"""

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    assert isinstance(data, dict), "expected {doc_id: text}"

    return {
        str(doc_id): (value["text"] if isinstance(value, dict) else value)
        for doc_id, value in data.items()
    }


with open(DATA_FILES["labels"], "r", encoding="utf-8") as f:
    LABELS = [line.strip() for line in f if line.strip()]

print(f"{len(LABELS)} labels:", LABELS)


def parse_annotation(annotation):
    """One gold entity -> (start, end, label, entity_text or None)."""

    if isinstance(annotation, dict):

        def pick(*names):
            for name in names:
                if name in annotation:
                    return annotation[name]
            raise KeyError(f"none of {names} in {annotation}")

        return (
            int(pick("start", "start_char", "begin")),
            int(pick("end", "end_char")),
            str(pick("label", "type", "tag")),
            annotation.get("text", annotation.get("entity", annotation.get("entity_text"))),
        )

    # list / tuple: [start, end, label, (entity_text)]
    entity_text = annotation[3] if len(annotation) > 3 else None

    return int(annotation[0]), int(annotation[1]), str(annotation[2]), entity_text


def load_gold(path, texts):
    """{doc_id: [[start, end, label], ...]} with end EXCLUSIVE."""

    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    assert isinstance(raw, dict), f"expected {{doc_id: [entities]}}, got {type(raw)}"

    parsed = {}

    for doc_id, annotations in raw.items():

        if isinstance(annotations, dict):
            annotations = annotations.get("entities", annotations.get("annotations", []))

        parsed[str(doc_id)] = [parse_annotation(a) for a in annotations]

    # ---- is the end offset exclusive (Python style) or inclusive? ----
    exclusive = inclusive = checked = 0

    for doc_id, entities in parsed.items():

        text = texts.get(doc_id)

        for start, end, label, entity_text in entities:

            if text is None or entity_text is None:
                continue

            checked += 1
            exclusive += text[start:end] == entity_text
            inclusive += text[start:end + 1] == entity_text

    shift = 0

    if checked == 0:
        print("  WARNING: gold has no entity text, cannot check the end-offset convention (assuming exclusive).")
    else:
        print(f"  end-offset check on {checked} entities: exclusive matches={exclusive}, inclusive matches={inclusive}")

        if inclusive > exclusive:
            shift = 1
            print("  -> end offsets are INCLUSIVE in the gold file; converting to exclusive.")

    gold = {}
    unknown_labels = Counter()

    for doc_id, entities in parsed.items():

        gold[doc_id] = []

        for start, end, label, entity_text in entities:

            if label not in LABELS:
                unknown_labels[label] += 1
                continue

            gold[doc_id].append([start, end + shift, label])

    if unknown_labels:
        print("  WARNING: labels not in labels.txt were ignored:", dict(unknown_labels))

    return gold, shift


texts = {}
gold = {}
END_SHIFT = 0

for fold in sorted(set(TRAIN_FOLDS + VAL_FOLDS)):

    print(f"\n{fold}")

    texts[fold] = load_texts(DATA_FILES[f"{fold}_text"])
    gold[fold], shift = load_gold(DATA_FILES[f"{fold}_gold"], texts[fold])

    END_SHIFT = max(END_SHIFT, shift)

    missing = set(texts[fold]) - set(gold[fold])
    print(f"  documents: {len(texts[fold])} | entities: {sum(len(v) for v in gold[fold].values())} | "
          f"docs without gold entry: {len(missing)}")

    for doc_id in missing:
        gold[fold][doc_id] = []

    if DEBUG_MODE:
        keep = list(texts[fold])[:DEBUG_DOCS]
        texts[fold] = {k: texts[fold][k] for k in keep}
        gold[fold] = {k: gold[fold][k] for k in keep}


test_texts = load_texts(DATA_FILES["test_text"])

if DEBUG_MODE:
    test_texts = {k: test_texts[k] for k in list(test_texts)[:DEBUG_DOCS]}

print(f"\nTest documents: {len(test_texts)}")

# sample so you can see the real format
first_fold = TRAIN_FOLDS[0]
sample_id = next(iter(gold[first_fold]))
sample_text = texts[first_fold][sample_id]

print("\nSample document", sample_id, "->", repr(sample_text[:120]))

for start, end, label in gold[first_fold][sample_id][:5]:
    print(f"  [{start}, {end}, {label}]  {sample_text[start:end]!r}")

12 labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']

fold1
  end-offset check on 4120 entities: exclusive matches=4120, inclusive matches=9
  documents: 35 | entities: 4120 | docs without gold entry: 0

fold2
  end-offset check on 4273 entities: exclusive matches=4273, inclusive matches=8
  documents: 35 | entities: 4273 | docs without gold entry: 0

Test documents: 35

  [137, 153, RESP]  'STATE OF HARYANA'
  [252, 261, DATE]  '19.8.2011'
  [276, 308, COURT]  'High Court of Punjab and Haryana'
  [359, 369, COURT]  'High Court'
  [575, 585, DATE]  '18.11.1994'


### CELL 6 · Tags (BIO), word features, spans, strict F1

**In simple words:** Turn entities into one tag per word — **B** = first word of an entity, **I** = inside, **O** = outside — and back. Also simple word features (capital letters, digits, punctuation) and the scoring rule.

**How it works:** 12 labels → 25 BIO tags. Word shape + 6 flags help with names and numbers. Strict F1 = start, end **and** label must all match (the Codabench rule), averaged over the labels (macro-F1). Viterbi decoding is defined here too.

In [6]:
# ============================================================
# CELL 6 — TAGS (BIO), WORD FEATURES, SPANS, STRICT F1
#
# Every WORD gets one of 25 tags:  O, B-<label>, I-<label>
#     B = first word of an entity   I = inside an entity
# ============================================================

TAGS = ["O"]

for label in LABELS:
    TAGS += [f"B-{label}", f"I-{label}"]

TAG2ID = {tag: i for i, tag in enumerate(TAGS)}
NUM_TAGS = len(TAGS)

# tag id -> ("B" / "I" / "O", label)
TAG_PARTS = [("O", None)] + [(tag[0], tag[2:]) for tag in TAGS[1:]]

# valid transitions: "I-X" may only follow "B-X" or "I-X"; a sequence cannot start with "I-X"
ALLOWED = np.zeros((NUM_TAGS, NUM_TAGS), dtype=bool)          # [previous tag, current tag]

for prev in range(NUM_TAGS):
    for cur in range(NUM_TAGS):
        prev_prefix, prev_label = TAG_PARTS[prev]
        cur_prefix, cur_label = TAG_PARTS[cur]
        ALLOWED[prev, cur] = cur_prefix != "I" or (prev_prefix != "O" and prev_label == cur_label)

START_ALLOWED = np.array([TAG_PARTS[t][0] != "I" for t in range(NUM_TAGS)])

print("Number of tags:", NUM_TAGS)


# ------------------------------------------------------------
# Word features: shape + 6 simple flags. They generalise across
# documents (a new name is still "Xxxx Xxxx", a date "dd.dd.dddd").
# ------------------------------------------------------------

def word_shape(word):

    shape = []

    for ch in word[:20]:
        if ch.isupper():
            c = "X"
        elif ch.islower():
            c = "x"
        elif ch.isdigit():
            c = "d"
        else:
            c = ch

        if len(shape) < 2 or not (shape[-1] == c and shape[-2] == c):   # "Xxxxxx" -> "Xxx"
            shape.append(c)

    return "".join(shape)


def word_flags(word):

    return [
        float(word.istitle()),
        float(word.isupper()),
        float(word.islower()),
        float(any(ch.isdigit() for ch in word)),
        float(word.isdigit()),
        float(not any(ch.isalnum() for ch in word)),
    ]


NUM_FLAGS = 6


# ------------------------------------------------------------
# Gold spans <-> word tags
# ------------------------------------------------------------

def first_non_space(text, start, end):

    while start < end and text[start].isspace():
        start += 1

    return start


def spans_to_tags(spans, text, starts, ends):
    """Gold entities -> one tag id per word."""

    n = len(starts)

    tags = np.zeros(n, dtype=np.int64)               # all "O"

    ends_arr = np.asarray(ends)
    trimmed_starts = np.array([first_non_space(text, s, e) for s, e in zip(starts, ends)], dtype=np.int64)

    dropped = 0

    for ent_start, ent_end, label in sorted(spans):

        first = int(np.searchsorted(ends_arr, ent_start, side="right"))
        last = int(np.searchsorted(trimmed_starts, ent_end, side="left")) - 1

        if last < first or first >= n or np.any(tags[first:last + 1] != 0):
            dropped += 1
            continue

        tags[first] = TAG2ID[f"B-{label}"]
        tags[first + 1:last + 1] = TAG2ID[f"I-{label}"]

    return tags, dropped


def tags_to_spans(tag_ids, text, starts, ends):
    """One tag per word -> [[start, end, label], ...] in original characters."""

    spans = []
    current_label = None
    current_start = current_end = 0

    def finish():

        if current_label is None:
            return

        s, e = current_start, current_end

        while s < e and text[s].isspace():
            s += 1

        while e > s and text[e - 1].isspace():
            e -= 1

        if e > s:
            spans.append([s, e, current_label])

    for i, tag_id in enumerate(tag_ids):

        prefix, label = TAG_PARTS[int(tag_id)]

        if prefix == "O":
            finish()
            current_label = None

        elif prefix == "B" or label != current_label:
            finish()
            current_label = label
            current_start, current_end = starts[i], ends[i]

        else:
            current_end = ends[i]

    finish()

    return spans


# ------------------------------------------------------------
# Viterbi: best valid tag sequence for a whole document
# ------------------------------------------------------------

def viterbi(emissions, transitions, start_scores, end_scores):
    """emissions [n_words, NUM_TAGS] -> best tag id per word (numpy)."""

    n = len(emissions)

    if n == 0:
        return np.zeros(0, dtype=np.int64)

    score = start_scores + emissions[0]
    backpointer = np.zeros((n, NUM_TAGS), dtype=np.int64)

    for i in range(1, n):
        candidates = score[:, None] + transitions          # [previous, current]
        backpointer[i] = candidates.argmax(axis=0)
        score = candidates.max(axis=0) + emissions[i]

    path = [int((score + end_scores).argmax())]

    for i in range(n - 1, 0, -1):
        path.append(int(backpointer[i, path[-1]]))

    return np.array(path[::-1], dtype=np.int64)


# ------------------------------------------------------------
# Strict macro P / R / F1 (start, end and label must all match)
# ------------------------------------------------------------

def strict_scores(gold_spans, predicted_spans):

    tp, fp, fn = Counter(), Counter(), Counter()

    for doc_id in gold_spans:

        gold_set = {(s, e, l) for s, e, l in gold_spans[doc_id]}
        pred_set = {(s, e, l) for s, e, l in predicted_spans.get(doc_id, [])}

        for _, _, label in gold_set & pred_set:
            tp[label] += 1
        for _, _, label in pred_set - gold_set:
            fp[label] += 1
        for _, _, label in gold_set - pred_set:
            fn[label] += 1

    per_label = {}

    for label in LABELS:

        if tp[label] + fp[label] + fn[label] == 0:
            continue

        precision = tp[label] / (tp[label] + fp[label]) if tp[label] + fp[label] else 0.0
        recall = tp[label] / (tp[label] + fn[label]) if tp[label] + fn[label] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        per_label[label] = {"precision": precision, "recall": recall, "f1": f1,
                            "support": tp[label] + fn[label]}

    macro = {
        key: float(np.mean([v[key] for v in per_label.values()])) if per_label else 0.0
        for key in ("precision", "recall", "f1")
    }

    return macro, per_label

Number of tags: 25


### CELL 7 · Tokenize all documents (tokens -> words) + windows

**In simple words:** Tokenize every document, remember which tokens belong to which word, and cut long documents into overlapping windows that fit the model.

**How it works:** Every word is predicted from its first token; a word inside several windows gets the average of their scores.

In [7]:
# ============================================================
# CELL 7 — TOKENIZE ALL DOCUMENTS (tokens -> words) + WINDOWS
# ============================================================

def make_doc(doc_id, text, spans=None):

    ids, starts, ends, words = encode_with_offsets(text)

    # A new "word" (tagging unit) starts at a token when
    #   - a new pre-tokenizer piece starts, or
    #   - letters meet digits inside a piece  ("Verma1" -> Verma | 1, "dated24" -> dated | 24)
    #   - the token starts with an apostrophe ("Board's" -> Board | 's)
    # because gold entities often end exactly there.
    new_word = np.ones(len(ids), dtype=bool)

    for i in range(1, len(ids)):

        if words[i] == words[i - 1]:

            previous_char = text[ends[i - 1] - 1]
            current_char = text[starts[i]]

            new_word[i] = (previous_char.isdigit() != current_char.isdigit()) or current_char in "'’"

    first_token = np.flatnonzero(new_word)
    last_token = np.r_[first_token[1:] - 1, len(ids) - 1] if len(ids) else first_token

    word_starts = [starts[i] for i in first_token]
    word_ends = [ends[i] for i in last_token]
    word_texts = [text[first_non_space(text, s, e):e] for s, e in zip(word_starts, word_ends)]

    doc = {
        "id": doc_id,
        "text": text,
        "ids": np.asarray(ids, dtype=np.int64),
        "first_token": first_token,
        "word_starts": word_starts,
        "word_ends": word_ends,
        "shapes": [word_shape(w) for w in word_texts],
        "flags": np.asarray([word_flags(w) for w in word_texts], dtype=np.float32).reshape(-1, NUM_FLAGS),
        "spans": spans,
        "word_tags": None,
        "dropped": 0,
    }

    if spans is not None:
        doc["word_tags"], doc["dropped"] = spans_to_tags(spans, text, word_starts, word_ends)

    return doc


def make_windows(n_tokens):
    """Start positions of overlapping windows that cover the whole document."""

    if n_tokens == 0:
        return []

    if n_tokens <= WINDOW:
        return [0]

    window_starts = list(range(0, n_tokens - WINDOW + 1, STRIDE))

    if window_starts[-1] != n_tokens - WINDOW:
        window_starts.append(n_tokens - WINDOW)

    return window_starts


start_time = time.time()

train_docs = [make_doc(i, t, gold[f][i]) for f in TRAIN_FOLDS for i, t in texts[f].items()]
val_docs = [make_doc(i, t, gold[f][i]) for f in VAL_FOLDS for i, t in texts[f].items()]
test_docs = [make_doc(i, t) for i, t in test_texts.items()]


# ------------------------------------------------------------
# Word-shape vocabulary (from the training documents; 0 = padding, 1 = unknown)
# ------------------------------------------------------------

shape_counts = Counter(s for d in train_docs for s in d["shapes"])

SHAPE2ID = {"<pad>": 0, "<unk>": 1}

for shape, count in shape_counts.most_common():
    if count >= 2:
        SHAPE2ID[shape] = len(SHAPE2ID)

for doc in train_docs + val_docs + test_docs:
    doc["shape_ids"] = np.array([SHAPE2ID.get(s, 1) for s in doc["shapes"]], dtype=np.int64)


# the token ids must match tokenizer.encode exactly
mismatch = 0

for doc in (train_docs + val_docs + test_docs)[:200]:
    if tok_module.normalize_text(doc["text"]) == doc["text"]:
        mismatch += doc["ids"].tolist() != tokenizer.encode(doc["text"])

assert mismatch == 0, f"{mismatch} documents differ from tokenizer.encode"

print(f"Tokenized in {time.time() - start_time:.1f}s | word shapes: {len(SHAPE2ID)}")

for name, docs in (("train", train_docs), ("val", val_docs), ("test", test_docs)):

    n_tokens = sum(len(d["ids"]) for d in docs)
    n_words = sum(len(d["first_token"]) for d in docs)
    n_windows = sum(len(make_windows(len(d["ids"]))) for d in docs)

    print(f"{name:5s}: {len(docs):4d} docs | {n_tokens:>9,} tokens | {n_words:>9,} words | {n_windows:>5,} windows")

print("Gold entities dropped while tagging (overlap / no word):", sum(d["dropped"] for d in train_docs + val_docs))

label_counts = Counter(l for f in TRAIN_FOLDS for spans in gold[f].values() for _, _, l in spans)

print("\nEntities per label in the training folds:")
for label in LABELS:
    print(f"  {label:10s} {label_counts[label]:6d}")

Tokenized in 10.5s | word shapes: 122
train:   35 docs |   462,556 tokens |   408,151 words | 1,211 windows
val  :   35 docs |   468,040 tokens |   414,107 words | 1,224 windows
test :   35 docs |   494,497 tokens |   438,417 words | 1,293 windows
Gold entities dropped while tagging (overlap / no word): 0

Entities per label in the training folds:
  APP           207
  RESP          191
  A.COUNSEL      70
  R.COUNSEL      87
  JUDGE         123
  WIT            67
  AUTH          535
  COURT         455
  STAT          525
  PREC          624
  DATE          813
  CASENO        423


### CELL 8 · Offset self-test (run before training!)

**In simple words:** A check without any model: turn the gold entities into word tags and back again. The score is the best any word-level tagger could ever reach here.

**How it works:** Expected ≈ 0.99 strict macro-F1; a clearly lower value would mean entity boundaries fall inside words (an offset bug).

In [8]:
# ============================================================
# CELL 8 — OFFSET SELF-TEST  (run before training!)
#
# gold entities -> word BIO tags -> spans back -> must equal gold.
# The score is the BEST F1 any model could ever reach with
# word-level tagging. If it is clearly below 1.0, some entity
# boundaries fall inside words.
# ============================================================

check_docs = val_docs if val_docs else train_docs
check_gold = {}
check_pred = {}

for doc in check_docs:

    check_gold[doc["id"]] = doc["spans"]
    check_pred[doc["id"]] = tags_to_spans(doc["word_tags"], doc["text"], doc["word_starts"], doc["word_ends"])

macro, per_label = strict_scores(check_gold, check_pred)

print(f"Round-trip (gold -> tags -> spans) strict macro-F1: {macro['f1']:.4f}")

for label, scores in per_label.items():
    if scores["f1"] < 0.999:
        print(f"  {label:10s} F1={scores['f1']:.4f}  (support {scores['support']})")

if macro["f1"] < 0.5:
    raise RuntimeError("Offsets look wrong (round-trip F1 < 0.5): check the end-offset convention and the sample in CELL 5.")

if macro["f1"] < 0.98:
    print("WARNING: some entity boundaries are inside words, so this is the maximum possible F1 for this setup.")
else:
    print("✅ Offsets are exact.")

CEILING_F1 = macro["f1"]

Round-trip (gold -> tags -> spans) strict macro-F1: 0.9938
  APP        F1=0.9900  (support 200)
  RESP       F1=0.9895  (support 190)
  AUTH       F1=0.9957  (support 468)
  COURT      F1=0.9785  (support 558)
  STAT       F1=0.9823  (support 735)
  PREC       F1=0.9893  (support 559)
✅ Offsets are exact.


### CELL 9 · Batches

**In simple words:** Group windows into training batches.

**How it works:** The model reads `WINDOW` tokens and predicts a tag for every word whose first token is in the window.

In [9]:
# ============================================================
# CELL 9 — BATCHES
#
# A window = WINDOW consecutive tokens. The model reads the tokens,
# then predicts one tag for every word whose FIRST token is inside
# the window.
# ============================================================

def window_words(doc, s):
    """Index range [w0, w1) of the words whose first token lies in the window starting at token s."""

    n = min(WINDOW, len(doc["ids"]) - s)

    w0 = int(np.searchsorted(doc["first_token"], s, side="left"))
    w1 = int(np.searchsorted(doc["first_token"], s + n, side="left"))

    return n, w0, w1


def make_batch(docs, items):
    """
    items = list of (doc_index, window_start).
    Returns a dict of tensors:
        ids, mask            [B, T]   tokens
        word_pos, word_mask  [B, W]   position of each word's first token in the window
        shape_ids            [B, W]
        flags                [B, W, NUM_FLAGS]
        labels               [B, W]   word tags (-1 = no label)
    """

    sizes = [window_words(docs[d], s) for d, s in items]

    T = max(n for n, _, _ in sizes)
    W = max(max(w1 - w0 for _, w0, w1 in sizes), 1)
    B = len(items)

    batch = {
        "ids": torch.full((B, T), PAD_ID, dtype=torch.long),
        "mask": torch.zeros((B, T), dtype=torch.bool),
        "word_pos": torch.zeros((B, W), dtype=torch.long),
        "word_mask": torch.zeros((B, W), dtype=torch.bool),
        "shape_ids": torch.zeros((B, W), dtype=torch.long),
        "flags": torch.zeros((B, W, NUM_FLAGS), dtype=torch.float32),
        "labels": torch.full((B, W), -1, dtype=torch.long),
    }

    for row, ((d, s), (n, w0, w1)) in enumerate(zip(items, sizes)):

        doc = docs[d]
        k = w1 - w0

        batch["ids"][row, :n] = torch.from_numpy(doc["ids"][s:s + n])
        batch["mask"][row, :n] = True

        batch["word_pos"][row, :k] = torch.from_numpy(doc["first_token"][w0:w1] - s)
        batch["word_mask"][row, :k] = True
        batch["shape_ids"][row, :k] = torch.from_numpy(doc["shape_ids"][w0:w1])
        batch["flags"][row, :k] = torch.from_numpy(doc["flags"][w0:w1])

        if doc["word_tags"] is not None and k > 0:

            tags = doc["word_tags"][w0:w1].copy()

            # window starts inside an entity: its first word becomes "B-" (an "I-" cannot start a sequence)
            if TAG_PARTS[tags[0]][0] == "I":
                tags[0] -= 1                                        # I-X -> B-X  (B-X is always the id before I-X)

            batch["labels"][row, :k] = torch.from_numpy(tags)

    return batch


def window_items(docs):
    """All (doc_index, window_start) pairs that contain at least one word."""

    return [
        (d, s)
        for d, doc in enumerate(docs)
        for s in make_windows(len(doc["ids"]))
        if window_words(doc, s)[2] > window_words(doc, s)[1]
    ]


example = make_batch(train_docs, window_items(train_docs)[:4])

print({k: tuple(v.shape) for k, v in example.items()})
print("Training windows:", len(window_items(train_docs)))

{'ids': (4, 512), 'mask': (4, 512), 'word_pos': (4, 456), 'word_mask': (4, 456), 'shape_ids': (4, 456), 'flags': (4, 456, 6), 'labels': (4, 456)}
Training windows: 1211


### CELL 10 · Model

**In simple words:** The network: the Phase-1 model reads the tokens, and a small head on top decides the tag of every word.

**How it works:** first-token vector of each word + 32-d word-shape embedding + 6 word flags → BiLSTM over the words (gives the causal decoder right-hand context) → CRF, which only allows valid tag sequences (no `I-` after `O`). Adaptation: full fine-tuning of the Architecture-2 encoder (all 6 layers + embeddings).

In [10]:
# ============================================================
# CELL 10 — MODEL
#
#   tokens -> Architecture-2 ENCODER (pretrained, bidirectional)
#          -> first-token vector of every word
#          + word-shape embedding + word flags
#          -> BiLSTM over the words
#          -> CRF (learns which tag follows which)
#
# The encoder layers have the same names as in the Architecture-2
# notebook, so the pretrained weights load directly.
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, attn_mask):

        q = self.split_heads(self.q_proj(x))
        k = self.split_heads(self.k_proj(x))
        v = self.split_heads(self.v_proj(x))

        # attn_mask: [B, 1, 1, T]  True = real token, False = padding
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
        )

        B, H, T, d = out.shape

        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * d))


class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, d_model, dropout):
        super().__init__()

        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        positions = torch.arange(x.size(1), device=x.device)

        return self.dropout(x + self.embedding(positions).unsqueeze(0))


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, attn_mask):

        x = x + self.self_attention(self.norm1(x), attn_mask)
        x = x + self.feed_forward(self.norm2(x))

        return x


class TransformerEncoder(nn.Module):

    def __init__(self, vocab_size, max_len, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = PositionalEmbedding(max_len, d_model, dropout)

        self.layers = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )

        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, input_ids, attn_mask):

        x = self.position_embedding(self.token_embedding(input_ids))

        for layer in self.layers:
            x = layer(x, attn_mask)

        return self.final_norm(x)


class CRF(nn.Module):
    """Linear-chain CRF. Invalid BIO transitions are fixed to -10000."""

    def __init__(self, num_tags):
        super().__init__()

        self.transitions = nn.Parameter(torch.zeros(num_tags, num_tags))      # [previous, current]
        self.start = nn.Parameter(torch.zeros(num_tags))
        self.end = nn.Parameter(torch.zeros(num_tags))

        self.register_buffer("invalid", torch.from_numpy(~ALLOWED))
        self.register_buffer("invalid_start", torch.from_numpy(~START_ALLOWED))

    def scores(self):

        transitions = self.transitions.masked_fill(self.invalid, -10000.0)
        start = self.start.masked_fill(self.invalid_start, -10000.0)

        return transitions, start, self.end

    def loss(self, emissions, tags, mask):
        """Mean negative log-likelihood per word. emissions [B, W, C], tags [B, W], mask [B, W]."""

        transitions, start, end = self.scores()

        B, W, C = emissions.shape
        tags = tags.clamp(min=0)
        maskf = mask.float()

        # score of the gold tag sequence
        gold = start[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, :1]).squeeze(1)

        for i in range(1, W):
            step = transitions[tags[:, i - 1], tags[:, i]] + emissions[:, i].gather(1, tags[:, i:i + 1]).squeeze(1)
            gold = gold + step * maskf[:, i]

        last_tag = tags.gather(1, (mask.sum(1) - 1).clamp(min=0).unsqueeze(1)).squeeze(1)
        gold = gold + end[last_tag]

        # log-sum of the scores of ALL tag sequences (forward algorithm)
        alpha = start + emissions[:, 0]

        for i in range(1, W):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + transitions.unsqueeze(0), dim=1) + emissions[:, i]
            alpha = torch.where(mask[:, i:i + 1], nxt, alpha)

        log_z = torch.logsumexp(alpha + end, dim=1)

        return (log_z - gold).sum() / mask.sum()


class EncoderBiLSTMCRF(nn.Module):

    def __init__(self, encoder, d_model, num_shapes):
        super().__init__()

        self.encoder = encoder

        self.shape_embedding = nn.Embedding(num_shapes, SHAPE_DIM, padding_idx=0)
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        self.lstm = nn.LSTM(
            input_size=d_model + SHAPE_DIM + NUM_FLAGS,
            hidden_size=LSTM_HIDDEN,
            batch_first=True,
            bidirectional=True,
        )

        self.emission = nn.Linear(2 * LSTM_HIDDEN, NUM_TAGS)
        self.crf = CRF(NUM_TAGS)

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("encoder.")]

    def emissions(self, batch):
        """Tag scores for every word in the batch: [B, W, NUM_TAGS] (float32)."""

        hidden = self.encoder(batch["ids"], batch["mask"][:, None, None, :])                 # [B, T, D]

        index = batch["word_pos"].unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        word_vectors = hidden.gather(1, index)                                               # [B, W, D]

        x = torch.cat(
            [word_vectors.float(), self.shape_embedding(batch["shape_ids"]), batch["flags"]],
            dim=-1,
        )

        x = self.dropout(x)

        lengths = batch["word_mask"].sum(1).clamp(min=1).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))

        return self.emission(self.dropout(out)).float()

    def loss(self, batch):

        with autocast():
            emissions = self.emissions(batch)

        return self.crf.loss(emissions.float(), batch["labels"], batch["word_mask"])

### CELL 11 · Build the model from the Phase-1 Architecture-2 checkpoint

**In simple words:** Create the network, load the trained Phase-1 weights into it, and add the adaptation (the part that will learn).

**How it works:** The layer names match the Phase-1 notebook, so the checkpoint loads directly; if it is not at the configured path, the notebook searches `/kaggle/input`. The printed counts show how many weights train. Here: full fine-tuning of the Architecture-2 encoder (all 6 layers + embeddings).

In [11]:
# ============================================================
# CELL 11 — BUILD THE MODEL FROM THE PHASE-1 ARCHITECTURE-2 CHECKPOINT
# ============================================================

if not CHECKPOINT_PATH.exists():

    found = sorted(Path("/kaggle/input").rglob("best_encoder_decoder.pt"))

    assert found, f"{CHECKPOINT_PATH} not found and no best_encoder_decoder.pt under /kaggle/input"

    CHECKPOINT_PATH = found[0]
    print("Checkpoint found at:", CHECKPOINT_PATH)


phase1 = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
phase1_config = phase1["config"]

# keep only the encoder weights ("encoder.xxx" -> "xxx")
PHASE1_ENCODER_WEIGHTS = {
    key[len("encoder."):]: value
    for key, value in phase1["model_state_dict"].items()
    if key.startswith("encoder.")
}

del phase1

print("Phase-1 config:", {k: phase1_config[k] for k in
      ("architecture", "vocab_size", "src_len", "d_model", "n_heads", "encoder_layers", "ffn_dim", "dropout")})

assert phase1_config["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert WINDOW <= phase1_config["src_len"], f"WINDOW must be <= {phase1_config['src_len']} (encoder positions)"


def build_model():
    """Fresh model: pretrained Phase-1 encoder + new head (called again for stage B)."""

    encoder = TransformerEncoder(
        vocab_size=phase1_config["vocab_size"],
        max_len=phase1_config["src_len"],
        d_model=phase1_config["d_model"],
        n_heads=phase1_config["n_heads"],
        n_layers=phase1_config["encoder_layers"],
        d_ff=phase1_config["ffn_dim"],
        dropout=phase1_config["dropout"],
    )

    encoder.load_state_dict(PHASE1_ENCODER_WEIGHTS, strict=True)      # every name and shape must match

    return EncoderBiLSTMCRF(encoder, phase1_config["d_model"], len(SHAPE2ID)).to(DEVICE)


# free GPU memory if this cell is re-run
for name in ["model", "optimizer", "scaler"]:
    globals().pop(name, None)

sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

model = build_model()

n_total = sum(p.numel() for p in model.parameters())
n_encoder = sum(p.numel() for p in model.encoder.parameters())

print(f"\nParameters: {n_total:,} total | encoder {n_encoder:,} | head {n_total - n_encoder:,}")
print("All parameters are trained (full fine-tuning).")

Checkpoint found at: /kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-2/architecture_2_encoder_decoder/best_encoder_decoder.pt
Phase-1 config: {'architecture': 'encoder_decoder', 'vocab_size': 32004, 'src_len': 1024, 'd_model': 768, 'n_heads': 12, 'encoder_layers': 6, 'ffn_dim': 3072, 'dropout': 0.1}

Parameters: 70,090,748 total | encoder 67,894,272 | head 2,196,476
All parameters are trained (full fine-tuning).


### CELL 12 · Optimizer + LR schedule

**In simple words:** Set up the optimizer (AdamW) and the learning-rate schedule: a short warm-up, then the rate slowly goes down.

**How it works:** Only the weights that should train are given to the optimizer; the new head gets a larger learning rate than the backbone.

In [12]:
# ============================================================
# CELL 12 — OPTIMIZER + LR SCHEDULE
# ============================================================

def make_optimizer(model, total_steps):

    encoder_decay = [p for p in model.encoder.parameters() if p.dim() >= 2]
    encoder_no_decay = [p for p in model.encoder.parameters() if p.dim() < 2]

    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_decay, "lr": LR_ENCODER, "weight_decay": WEIGHT_DECAY},
            {"params": encoder_no_decay, "lr": LR_ENCODER, "weight_decay": 0.0},
            {"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.0},
        ],
        betas=(0.9, 0.999),
    )

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        """Linear warmup, then linear decay to 0."""

        if step < warmup_steps:
            factor = (step + 1) / warmup_steps
        else:
            factor = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        for group, base_lr in zip(optimizer.param_groups, base_lrs):
            group["lr"] = base_lr * factor

    return optimizer, set_lr

### CELL 13 · Prediction + evaluation

**In simple words:** Predict the entities of whole documents and score them.

**How it works:** Scores of overlapping windows are averaged per word, the CRF's Viterbi picks the best valid tag sequence for the whole document, tags become character spans, and strict macro-F1 compares them with gold.

In [13]:
# ============================================================
# CELL 13 — PREDICTION + EVALUATION
#
# Every window gives tag scores for its words. Words that appear in
# several windows get the average score. Then the CRF Viterbi picks
# the best valid tag sequence for the whole document.
# ============================================================

@torch.no_grad()
def predict_spans(model, docs):
    """Returns {doc_id: [[start, end, label], ...]}"""

    model.eval()

    score_sum = [np.zeros((len(d["first_token"]), NUM_TAGS), dtype=np.float32) for d in docs]
    counts = [np.zeros(len(d["first_token"]), dtype=np.float32) for d in docs]

    items = window_items(docs)

    for b in range(0, len(items), EVAL_BATCH_SIZE):

        batch_items = items[b:b + EVAL_BATCH_SIZE]
        batch = {k: v.to(DEVICE) for k, v in make_batch(docs, batch_items).items()}

        with autocast():
            emissions = model.emissions(batch)

        emissions = emissions.float().cpu().numpy()

        for row, (d, s) in enumerate(batch_items):

            _, w0, w1 = window_words(docs[d], s)

            score_sum[d][w0:w1] += emissions[row, :w1 - w0]
            counts[d][w0:w1] += 1

    transitions, start, end = (t.detach().float().cpu().numpy() for t in model.crf.scores())

    predictions = {}

    for d, doc in enumerate(docs):

        word_scores = score_sum[d] / np.maximum(counts[d], 1)[:, None]
        word_tags = viterbi(word_scores, transitions, start, end)

        predictions[doc["id"]] = tags_to_spans(word_tags, doc["text"], doc["word_starts"], doc["word_ends"])

    model.train()

    return predictions


def evaluate(model, docs):

    gold_spans = {doc["id"]: doc["spans"] for doc in docs}

    return strict_scores(gold_spans, predict_spans(model, docs))

### CELL 14 · Training function + stage A (train fold1, validate fold2)

**In simple words:** **Stage A:** train on fold 1 and check on fold 2 after every epoch; keep the best epoch.

**How it works:** Stage A gives the validation score for the report and the number of epochs used in stage B.

In [14]:
# ============================================================
# CELL 14 — TRAINING FUNCTION + STAGE A (train fold1, validate fold2)
# ============================================================

def train_model(model, docs, epochs, val_docs=None, save_path=None, patience=None):
    """Trains `model` on `docs`. With val_docs: keeps the best epoch (by strict macro-F1)."""

    items = window_items(docs)
    steps_per_epoch = math.ceil(len(items) / BATCH_SIZE)

    optimizer, set_lr = make_optimizer(model, steps_per_epoch * epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    rng = np.random.default_rng(SEED)
    history = []
    best_f1, best_epoch, step = -1.0, 0, 0
    start = time.time()

    model.train()

    for epoch in range(1, epochs + 1):

        order = rng.permutation(len(items))
        epoch_loss = 0.0

        for b in range(0, len(order), BATCH_SIZE):

            batch_items = [items[i] for i in order[b:b + BATCH_SIZE]]
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in make_batch(docs, batch_items).items()}

            set_lr(step)

            loss = model.loss(batch)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            step += 1
            epoch_loss += loss.item()

        row = {"epoch": epoch, "train_loss": epoch_loss / steps_per_epoch,
               "train_f1": evaluate(model, docs[:TRAIN_EVAL_DOCS])[0]["f1"]}

        line = (f"Epoch {epoch:2d}/{epochs} | loss={row['train_loss']:.4f} "
                f"train F1={row['train_f1']:.4f}")

        if val_docs:

            macro, _ = evaluate(model, val_docs)
            row.update(val_f1=macro["f1"], val_precision=macro["precision"], val_recall=macro["recall"])
            line += f" | val F1={macro['f1']:.4f} P={macro['precision']:.4f} R={macro['recall']:.4f}"

            if macro["f1"] > best_f1:
                best_f1, best_epoch = macro["f1"], epoch
                torch.save(model.state_dict(), save_path)
                line += " ⭐ best"

        history.append(row)
        print(line + f" | {(time.time() - start) / 60:.1f} min", flush=True)

        if val_docs and patience and epoch - best_epoch >= patience:
            print(f"No improvement for {patience} epochs -> early stop.")
            break

    if not val_docs and save_path:
        torch.save(model.state_dict(), save_path)
        best_epoch = epochs

    return history, best_epoch, best_f1, (time.time() - start) / 60


print("=" * 90)
print("STAGE A: train", TRAIN_FOLDS, "| validate", VAL_FOLDS)
print("=" * 90)

history_a, best_epoch, best_f1, minutes_a = train_model(
    model, train_docs, EPOCHS, val_docs=val_docs, save_path=BEST_MODEL_PATH, patience=PATIENCE,
)

print(f"\n✅ Stage A done in {minutes_a:.1f} min | best epoch {best_epoch} | "
      f"best val strict macro-F1 {best_f1:.4f} (ceiling {CEILING_F1:.4f})")

STAGE A: train ['fold1'] | validate ['fold2']
Epoch  1/15 | loss=0.4256 train F1=0.1227 | val F1=0.0806 P=0.2023 R=0.0565 ⭐ best | 1.5 min
Epoch  2/15 | loss=0.0424 train F1=0.3360 | val F1=0.2108 P=0.2491 R=0.1902 ⭐ best | 2.9 min
Epoch  3/15 | loss=0.0258 train F1=0.5262 | val F1=0.2540 P=0.2729 R=0.2516 ⭐ best | 4.3 min
Epoch  4/15 | loss=0.0183 train F1=0.6945 | val F1=0.2802 P=0.3927 R=0.2429 ⭐ best | 5.7 min
Epoch  5/15 | loss=0.0125 train F1=0.7450 | val F1=0.2978 P=0.3565 R=0.2777 ⭐ best | 7.2 min
Epoch  6/15 | loss=0.0095 train F1=0.8116 | val F1=0.2896 P=0.3845 R=0.2584 | 8.6 min
Epoch  7/15 | loss=0.0069 train F1=0.8519 | val F1=0.3070 P=0.3765 R=0.2784 ⭐ best | 10.0 min
Epoch  8/15 | loss=0.0048 train F1=0.9059 | val F1=0.3317 P=0.3795 R=0.3055 ⭐ best | 11.4 min
Epoch  9/15 | loss=0.0038 train F1=0.9304 | val F1=0.2976 P=0.3994 R=0.2569 | 12.8 min
Epoch 10/15 | loss=0.0029 train F1=0.9418 | val F1=0.3219 P=0.3949 R=0.2900 | 14.3 min
Epoch 11/15 | loss=0.0021 train F1=0.9723

### CELL 15 · Stage a report (per label) + save results

**In simple words:** Scores per label on fold 2 (for the report), saved to `results.json`.

In [15]:
# ============================================================
# CELL 15 — STAGE A REPORT (per label) + SAVE RESULTS
# ============================================================

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))

macro, per_label = evaluate(model, val_docs)

print(f"Validation ({VAL_FOLDS}) strict macro: F1={macro['f1']:.4f}  "
      f"P={macro['precision']:.4f}  R={macro['recall']:.4f}\n")

print(f"{'label':10s} {'P':>7s} {'R':>7s} {'F1':>7s} {'support':>8s}")

for label, s in per_label.items():
    print(f"{label:10s} {s['precision']:7.3f} {s['recall']:7.3f} {s['f1']:7.3f} {s['support']:8d}")

results = {
    "experiment": "E1 - Architecture-2 encoder (full fine-tuning) + word features + BiLSTM + CRF",
    "parameters": n_total,
    "encoder_parameters": n_encoder,
    "window": WINDOW,
    "stride": STRIDE,
    "best_epoch": best_epoch,
    "stage_a_minutes": minutes_a,
    "round_trip_ceiling_f1": CEILING_F1,
    "val_macro": macro,
    "val_per_label": per_label,
    "history": history_a,
}

with open(OUTPUT_DIR / "e1_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", OUTPUT_DIR / "e1_results.json")

Validation (['fold2']) strict macro: F1=0.3317  P=0.3795  R=0.3055

label            P       R      F1  support
APP          0.146   0.060   0.085      200
RESP         0.049   0.026   0.034      190
A.COUNSEL    0.295   0.144   0.194      125
R.COUNSEL    0.404   0.216   0.282       97
JUDGE        0.303   0.189   0.232      122
WIT          0.000   0.000   0.000      120
AUTH         0.439   0.229   0.301      468
COURT        0.739   0.815   0.775      558
STAT         0.629   0.590   0.609      735
PREC         0.178   0.161   0.169      559
DATE         0.901   0.846   0.873      762
CASENO       0.473   0.389   0.427      337

Saved: /kaggle/working/e1_ner_arch2/e1_results.json


### CELL 16 · Stage B (fold1 + fold2) + test predictions -> submission.zip

**In simple words:** **Stage B:** retrain on fold 1 + fold 2 together (more data) for the best number of epochs, then predict the test fold and write `submission.zip`.

**How it works:** `prediction.json` = for every test document its list of `[start, end, label]` spans, at the root of the zip.

In [16]:
# ============================================================
# CELL 16 — STAGE B (fold1 + fold2) + TEST PREDICTIONS -> submission.zip
# ============================================================

if FINAL_TRAIN_ON_ALL_FOLDS:

    print("=" * 90)
    print(f"STAGE B: retrain on {TRAIN_FOLDS + VAL_FOLDS} for {best_epoch} epochs (best epoch of stage A)")
    print("=" * 90)

    del model
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    torch.manual_seed(SEED)

    model = build_model()

    _, _, _, minutes_b = train_model(model, train_docs + val_docs, max(1, best_epoch), save_path=FINAL_MODEL_PATH)

    print(f"Stage B done in {minutes_b:.1f} min")

# (otherwise `model` is the best stage-A model loaded in CELL 15)

start = time.time()
submission = predict_spans(model, test_docs)
print(f"\nPredicted {len(submission)} test documents in {time.time() - start:.1f}s")


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

assert set(submission) == set(test_texts), "prediction IDs must equal the official test IDs"

n_entities = 0

for doc_id, spans in submission.items():
    for start_char, end_char, label in spans:
        assert 0 <= start_char < end_char <= len(test_texts[doc_id]), (doc_id, start_char, end_char)
        assert label in LABELS, label
        n_entities += 1

print(f"✅ Valid: {len(submission)} documents, {n_entities} entities "
      f"({n_entities / max(1, len(submission)):.1f} per document)")
print("Predicted entities per label:", dict(Counter(l for s in submission.values() for _, _, l in s)))


# ------------------------------------------------------------
# Save  (end offsets in the SAME convention as the gold files)
# ------------------------------------------------------------

if END_SHIFT:
    print("Gold files use inclusive end offsets -> writing inclusive ends (end - 1).")

output = {
    doc_id: [[start_char, end_char - END_SHIFT, label] for start_char, end_char, label in spans]
    for doc_id, spans in submission.items()
}

PREDICTION_FILE = OUTPUT_DIR / "prediction.json"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False)

ZIP_FILE = OUTPUT_DIR / "submission.zip"

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("Zip contents:", zf.namelist(), "->", ZIP_FILE)

first_id = next(iter(output))
first_text = test_texts[first_id]

print(f"\nExample (document {first_id}):")
for start_char, end_char, label in output[first_id][:8]:
    print(f"  [{start_char}, {end_char}, {label!r}]  {first_text[start_char:end_char + END_SHIFT]!r}")

STAGE B: retrain on ['fold1', 'fold2'] for 8 epochs (best epoch of stage A)
Epoch  1/8 | loss=0.2370 train F1=0.3193 | 2.5 min
Epoch  2/8 | loss=0.0280 train F1=0.4510 | 5.0 min
Epoch  3/8 | loss=0.0182 train F1=0.6394 | 7.4 min
Epoch  4/8 | loss=0.0127 train F1=0.8042 | 9.9 min
Epoch  5/8 | loss=0.0086 train F1=0.8608 | 12.4 min
Epoch  6/8 | loss=0.0064 train F1=0.9089 | 14.9 min
Epoch  7/8 | loss=0.0047 train F1=0.9268 | 17.3 min
Epoch  8/8 | loss=0.0038 train F1=0.9394 | 19.8 min
Stage B done in 19.8 min

Predicted 35 test documents in 10.3s
✅ Valid: 35 documents, 3473 entities (99.2 per document)
Predicted entities per label: {'COURT': 581, 'CASENO': 259, 'RESP': 100, 'JUDGE': 88, 'DATE': 700, 'PREC': 526, 'AUTH': 222, 'STAT': 775, 'APP': 54, 'R.COUNSEL': 75, 'A.COUNSEL': 45, 'WIT': 48}
Zip contents: ['prediction.json'] -> /kaggle/working/e1_ner_arch2/submission.zip

Example (document 1128835):
  [18, 40, 'COURT']  'SUPREME COURT OF INDIA'
  [70, 98, 'CASENO']  'CIVIL APPEAL NO.658